### Pitching Moment Equation Rigid Body

![Alt text](https://www.researchgate.net/profile/Eric-Tuegel-2/publication/268478621/figure/fig1/AS:1137249819344897@1648152894317/Aircraft-Body-Frame-of-Reference.png)

The pitching moment equation for a rigid is given by:

$$
I_y \dot{q} - (I_z - I_x) rp - I_{xz} \left( \dot{p} - qr \right) = \vec{M}_y^{\text{CG}}
$$

Where the aerodynamic pitching moment $ M_y $ is expressed as:

$$
M_y = \frac{1}{2} \rho V^2 S c \left( C_{m_0} + C_{m_\alpha} \alpha + C_{m_{\delta_e}} \delta_e + C_{m_q} \frac{q c}{2 V} \right)
$$

### Soft kite
These equations can be used as is for a fixed wing kite. However, for a soft wing kite, the aerodynamic pitching moment is given by:

$$
\vec{M}_y^{\text{b}} = \vec{M}_y^{\text{CG}} + \sum_i \left( z_{i} F_{x,i} - x_{i} F_{z,i} \right)
$$

for i:

- Aerodynamic forces: $  z_{ca} F_{x,a} -  x_{ca} F_{z,a} $
- Gravity: $  z_{cg} W_{x,g} -  x_{cg} W_{z,g} $
- Inertial forces: $  z_{cg} m a_{x} -  x_{cg} m a_{z} $


Reorganizing and defining new aerodynamic moment coefficients at the bridle we have:

$$
\vec{M}_y^{\text{b}} = \frac{1}{2} \rho V^2 S c \left( C_{m_0} + C_{m_\alpha} \alpha_w + C_{m_{u_p}} u_p + C_{m_q} \frac{q c}{2 V} \right) + \sum_i \left( z_{i} F_{x,i} - x_{i} F_{z,i} \right)
$$

Assuming that the actuation $u_p$ is equivalent to an angle of attack change we can write:

$$
\theta_t = k_p u_p \quad \text{and} \quad \theta_t = -(\lambda_0 + \alpha_d)
$$

where $k_p$ is the pitch actuation gain and $\theta_t$ is the tether angle.

$$
\vec{M}_y^{\text{b}} = \frac{1}{2} \rho V^2 S c \left( C_{m_0} + C_{m_\alpha} (\alpha_t + \theta_t) + C_{m_q} \frac{q c}{2 V} \right) +\sum_i \left( z_{i} F_{x,i} - x_{i} F_{z,i} \right)
$$

---

### Trim Condition

$$
\frac{1}{2} \rho V^2 S c \left( C_{m_0} + C_{m_\alpha} \alpha_w \right) + z_{cg} W_{x} -  x_{cg} W_{z} + z_{ca} F_{a,x} -  x_{ca} F_{a,z}  = 0
$$

$$
\frac{1}{2} \rho V^2 S c \left( C_{m_0} + C_{m_\alpha} \cdot (\alpha_t +\theta_t) \right) + z_{cg} W_{x} -  x_{cg} W_{z} + z_{ca} F_{a,x} -  x_{ca} F_{a,z} = 0
$$

$\lambda_0$ is such that the kite is trimmed at the angle of attack given by the force equilibrium ??

### Force equilibrium

$$
\frac{1}{2} \rho V^2 S (C_L(\alpha_w) \sin{\alpha_t} - C_D(\alpha_w) \cos{\alpha_t}) - W_x  = 0
$$

In [41]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, FloatSlider

# Constants
Cm0 = 0.1
Cm_alpha = -2.5
alpha_d = 0
z_cg = -10
x_cg = 0
z_ca = -10
x_ca = 0
m_w = 15
m_kcu = 26
S = 20
c = 2
va = 30
rho = 1.225


# Function to find the zero crossing
def find_moment_zero_crossing(x, y):
    zero_crossings = np.where(np.diff(np.sign(y)))[0]
    if zero_crossings.size > 0:
        for idx in zero_crossings:
            if y[idx] > y[idx + 1]:
                return x[idx]
        idx = zero_crossings[0]
        return x[idx]
    return None  # No zero crossing

# Function to find the zero crossing
def find_force_zero_crossing(x, y):
    zero_crossings = np.where(np.diff(np.sign(y)))[0]
    if zero_crossings.size > 0:
        for idx in zero_crossings:
            if y[idx] < y[idx + 1]:
                return x[idx]
        idx = zero_crossings[0]
        return x[idx]
    return None  # No zero crossing


# Function to update plots based on lambda_0
def update_plot(theta_t_deg, course, x_ca, z_ca, theta_k):
    theta_t = np.radians(theta_t_deg)  # Convert slider value to radians
    course = np.radians(course)  # Convert slider value to radians
    theta_k = np.radians(theta_k)  # Convert slider value to radians
    alpha_t = np.linspace(np.radians(0), np.radians(40), 1000)
    alpha_w = alpha_t + theta_t + theta_k

    Wx = -m_w * np.cos(course) * 9.81

    Wx_kcu = -m_kcu * np.cos(course) * 9.81
    # Cm = 0.5 * rho * va**2 * S * c * (Cm0 + Cm_alpha * alpha_w) + z_cg * Wx

    CL = np.pi * alpha_w
    CD = 0.05 + 0.205 * CL**2

    Ftan_aero = (
        0.5 * rho * va**2 * S * (CL * np.sin(alpha_t) - CD * np.cos(alpha_t)) 
    )
    Ft_aero = 0.5 * rho * va**2 * S * (CL * np.cos(alpha_t) + CD * np.sin(alpha_t)) 
    Cm_aero = 0# 0.5 * rho * va**2 * S * c * (Cm0 + Cm_alpha * alpha_w)
    Cm =  Cm_aero + z_ca * Ftan_aero*np.cos(theta_k) + z_ca* Ft_aero*np.sin(theta_k) + z_cg * Wx * np.cos(theta_k)  + x_ca * Ftan_aero*np.sin(theta_k) + x_ca* Ft_aero*np.cos(theta_k) 


    Ftan = Ftan_aero + Wx_kcu + Wx
    # Find single zero crossings
    zero_crossing_Cm = find_moment_zero_crossing(alpha_w, Cm)
    zero_crossing_Ftan = find_force_zero_crossing(alpha_w, Ftan)


    # Plotting
    plt.figure(figsize=(8, 6))
    plt.plot(np.degrees(alpha_w), Cm, label="Pitching Moment, My")
    plt.plot(np.degrees(alpha_w), Ftan, label="Tangent Force, Ftan")

    # Plot vertical lines for zero crossings
    if zero_crossing_Cm is not None:
        plt.axvline(
            np.degrees(zero_crossing_Cm),
            color="blue",
            linestyle="--",
            label=f"Zero Crossing (Cm) at {np.degrees(zero_crossing_Cm):.2f}°",
        )
    if zero_crossing_Ftan is not None:
        plt.axvline(
            np.degrees(zero_crossing_Ftan),
            color="orange",
            linestyle="--",
            label=f"Zero Crossing (Ftan) at {np.degrees(zero_crossing_Ftan):.2f}°",
        )
    plt.xlabel("alpha_w [deg]")
    plt.ylabel("Values")
    plt.title(f"Plots with lambda_0 = {theta_t_deg}° and course = {np.degrees(course)}°")
    plt.grid()
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(np.degrees(alpha_w), Ft_aero, label="Tether Force, Ft")
    plt.plot(np.degrees(alpha_w), Ftan, label="Tangent Force, Ftan")
    if zero_crossing_Ftan is not None:
        plt.axvline(
            np.degrees(zero_crossing_Ftan),
            color="orange",
            linestyle="--",
            label=f"Zero Crossing (Ftan) at {np.degrees(zero_crossing_Ftan):.2f}°",
        )
    plt.xlabel("alpha_w [deg]")
    plt.ylabel("Values")
    plt.grid()
    plt.show()
# Slider widget
interact(
    update_plot,
    theta_t_deg=FloatSlider(value=-5, min=-20, max=20, step=1, description="theta_t (°)"),
    course=FloatSlider(value=90, min=0, max=360, step=1, description="course (°)"),
    x_ca=FloatSlider(value=0, min=-10, max=10, step=0.1, description="x_ca"),
    z_ca=FloatSlider(value=-10, min=-20, max=0, step=1, description="z_ca"),
    theta_k=FloatSlider(value=0, min=-10, max=10, step=0.1, description="theta_k"),
)


interactive(children=(FloatSlider(value=-5.0, description='theta_t (°)', max=20.0, min=-20.0, step=1.0), Float…

<function __main__.update_plot(theta_t_deg, course, x_ca, z_ca, theta_k)>